In [1]:
# Load API key từ .env hoặc nhập thủ công
import logging
import os
from pathlib import Path
from dotenv import load_dotenv

# Load từ stock_analysis/.env nếu có
env_path = Path("stock_analysis/.env")
if env_path.exists():
    load_dotenv(env_path)
    logging.info(f"✓ Loaded .env from {env_path}")
else:
    # Nhập thủ công nếu chưa có .env
    from getpass import getpass
    api_key = getpass("Nhập API_KEY: ")
    env_path.write_text(f"API_KEY={api_key}\n")
    os.environ["API_KEY"] = api_key
    logging.info(f"✓ API_KEY set manually, written to {env_path}")

assert os.environ.get("API_KEY"), "API_KEY chưa được set!"
logging.info("✓ API_KEY ready")

In [2]:
import time
import logging
import yaml
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from src.features.preprocessor import load_csv
from stock_analysis import pipeline

logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(levelname)s %(message)s', datefmt='%H:%M:%S')

# Load config
with open('configs/config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

WINDOW = cfg['env']['window']  # 20
MODEL = cfg['analysis']['model']
PATHS = cfg['data']['paths']

print(f'Window: {WINDOW}')
print(f'Model:  {MODEL}')
print(f'Stocks: {[Path(p).stem for p in PATHS]}')

# Kiểm tra data files
for p in PATHS:
    if Path(p).exists():
        df = load_csv(p)
        print(f"  ✓ {Path(p).stem}: {len(df)} rows | {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()} | Windows: {len(df)-WINDOW}")
    else:
        logging.error(f"  ✗ {p} NOT FOUND!")

Loaded as API: https://00f67cfc8f6a913580.gradio.live/
Loaded as API: https://cf4a4cd119eb8adcac.gradio.live/
Window: 20
Model:  unsloth/Qwen2.5-14B-Instruct-bnb-4bit
Stocks: ['VNM', 'FPT', 'VIC', 'HPG']
[Data] VNM.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VNM.csv Close range: 47.97 → 88.46
  ✓ VNM: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266
[Data] FPT.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] FPT.csv Close range: 22.91 → 131.67
  ✓ FPT: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266
[Data] VIC.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VIC.csv Close range: 19.95 → 169.90
  ✓ VIC: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266
[Data] HPG.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] HPG.csv Close range: 9.00 → 32.65
  ✓ HPG: 1286 rows | 2020-11-09 → 2025-12-31 | Windows: 1266


In [3]:
def warmup_stock(
    stock_id: str,
    csv_path: str,
    window: int = 20,
    skip_every: int = 1,
):
    """
    Cache tất cả windows cho 1 stock, xử lý tuần tự từng window một.
    
    Args:
        skip_every: Cache mỗi N windows (1 = tất cả, 5 = mỗi 5 ngày).
    """
    df = load_csv(csv_path)
    
    success_count = 0
    errors = 0
    error_msgs = []
    
    steps = list(range(window, len(df), skip_every))
    pbar = tqdm(steps, desc=stock_id, unit="win")
    
    for step in pbar:
        start_idx = max(0, step - window + 1)
        date_end = pd.Timestamp(df['date'].iloc[step]).to_pydatetime()
        date_start = pd.Timestamp(df['date'].iloc[start_idx]).to_pydatetime()
        
        try:
            pipeline(
                model=MODEL,
                stock_id=stock_id,
                date_start=date_start,
                date_end=date_end,
            )
            success_count += 1
        except Exception as e:
            logging.error(e)
            errors += 1
            if len(error_msgs) < 5:
                error_msgs.append(f"date={date_end.date()}: {e}")
        
        pbar.set_postfix(ok=success_count, err=errors)
    
    pbar.close()
    
    # Tóm tắt
    print(f"  ✓ {stock_id}: {success_count} ok, {errors} errors (total: {len(steps)})")
    if error_msgs:
        print(f"  Errors (showing first {len(error_msgs)}):")
        for msg in error_msgs:
            print(f"    • {msg}")
    
    return success_count, errors

In [4]:
warmup_stock('VIC', 'data/VIC.csv', WINDOW, skip_every=1)

[Data] VIC.csv: 1286 rows | 2020-11-09 → 2025-12-31
[Data] VIC.csv Close range: 19.95 → 169.90


VIC:   0%|          | 0/1266 [00:00<?, ?win/s]

For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


  ✓ VIC: 1266 ok, 0 errors (total: 1266)


(1266, 0)